# 6단계. 검색 (A: 벡터 검색)

- 저장은 `chunktest.ipynb`에서 끝났다. 여기서는 PGVector 서랍에 **연결만** 해서 검색한다.
- 서랍: `jh_500`(500자/겹침 100), `jh_1000`(1000자/겹침 200) — v1 데이터 + 법령 보강(2026-11-24, 조문 112 → 157)

## 6-1. DB의 서랍 두 개 불러오기

- ⚠️ `pre_delete_collection=True`는 절대 넣지 않는다. 넣으면 서랍이 비워진다.
- ⚠️ 서랍 이름을 틀리게 쓰면 오류 없이 빈 서랍이 새로 생긴다.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

load_dotenv()

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
connection_url = os.getenv("DATABASE_URL")

store_500 = PGVector(
    embeddings=embedding_model,
    connection=connection_url,
    collection_name="jh_500",
    use_jsonb=True,
)

store_1000 = PGVector(
    embeddings=embedding_model,
    connection=connection_url,
    collection_name="jh_1000",
    use_jsonb=True,
)

## 6-2. 질문 하나 던져 보기 (작동 확인)

- `similarity_search_with_score(질문, k=5)`: 질문과 가까운 조각 **5개**를 거리와 함께 가져온다. 5개인 이유는 발제 지표가 **Hit@5**이기 때문.
- 숫자는 **코사인 거리**: 0에 가까울수록 질문과 비슷하다. (유사도 = 1 − 거리)

In [2]:
question = "전입신고는 언제까지 해야 하나요?"

results = store_500.similarity_search_with_score(question, k=5)

for doc, score in results:
    print(f"{score:.4f}", doc.metadata["chunk_id"], doc.page_content[:60].replace("\n", " "))

0.4460 YOUTH_6568_p004_s500_c06 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 후 (입주 후 7일
0.4675 YOUTH_6448_p008_s500_c01 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 후  ★전입신고 후
0.4675 YOUTH_6535_p008_s500_c01 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 후  ★전입신고 후
0.4686 YOUTH_6539_p008_s500_c01 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 후  ★전입신고 후
0.4686 YOUTH_6590_p009_s500_c01 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 후  ★전입신고 후


## 6-3. 질문 몇 개로 500 vs 1000 비교 (작동 확인)

- ⚠️ 아래 질문은 **작동 확인용**이다. 평가셋(dev/test)이 아니다. 평가셋의 정답·근거는 원문을 보고 따로 적는다.
- 마지막 질문(전세사기)은 데이터에 근거가 약한 주제(AtoZ 전자책 미수집) — 근거가 약할 때 거리가 어떻게 나오는지 보려고 넣었다.
- 결과가 나쁘게 나와도 지우지 않는다(발제: 실패·악화 결과도 남긴다).

In [3]:
questions = [
    "전입신고는 언제까지 해야 하나요?",
    "월세 계약하면 따로 신고해야 하나요?",
    "이사 나갈 때 원상복구는 어디까지 해야 하나요?",
    "행복주택 청년 자격이 어떻게 되나요?",
    "전세사기 안 당하려면 계약 전에 뭘 확인해야 하나요?",
]

stores = {"jh_500": store_500, "jh_1000": store_1000}

for q in questions:
    print("=" * 100)
    print("질문:", q)
    for name, store in stores.items():
        print(f"  [{name}]")
        for rank, (doc, score) in enumerate(store.similarity_search_with_score(q, k=5), start=1):
            where = doc.metadata.get("article") or f"p{doc.metadata.get('page')}"
            print(f"   {rank}. {score:.4f} | {doc.metadata['doc_id']} {where} | {doc.page_content[:50].replace(chr(10), ' ')}")

질문: 전입신고는 언제까지 해야 하나요?
  [jh_500]


   1. 0.4460 | YOUTH_6568 p4 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   2. 0.4675 | YOUTH_6448 p8 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   3. 0.4675 | YOUTH_6535 p8 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   4. 0.4686 | YOUTH_6539 p8 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   5. 0.4686 | YOUTH_6590 p9 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
  [jh_1000]


   1. 0.4460 | YOUTH_6568 p4 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   2. 0.4675 | YOUTH_6448 p8 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   3. 0.4675 | YOUTH_6535 p8 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   4. 0.4686 | YOUTH_6539 p8 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
   5. 0.4686 | YOUTH_6590 p9 | 19. 그 외 자격확인을 위하여 사업주체가 요구하는 서류  ★입주 당일 전입  신고를 한 
질문: 월세 계약하면 따로 신고해야 하나요?
  [jh_500]


   1. 0.4831 | SH_302368 p4 | ■ 계약방식 : 전세 또는 보증부 월세 방식 중 한 가지 선택 전세 보증금한도액 이내의 주
   2. 0.5129 | GUIDE_SEOUL_CASES p23 | 전세 계약하려는 집이 매매 진행 중입니다.  누구와 계약해야 할까요?  02  대리인 계약
   3. 0.5193 | GUIDE_SEOUL_HLPA_2020 p8 | 두 번째는 ‘전월세 상한제’야, 보통 집주인이 계약만기 때 보증금이나 월세를 올려 받는 경
   4. 0.5249 | SH_308340 p46 | ‘여권정보증명서’를 지참하여야 함  ‘여권정보증명서’를 지참하여야 함  ※  계약금을 납부
   5. 0.5270 | GUIDE_SEOUL_HLPA_2020 p35 | Q14 집주인이 전세 계약을 월세 계약으로 바꾸자고 해요 요즘 뉴스에 보니까 집주인들이 전
  [jh_1000]


   1. 0.5128 | GUIDE_SEOUL_CASES p23 | 전세 계약하려는 집이 매매 진행 중입니다.  누구와 계약해야 할까요?  02  대리인 계약
   2. 0.5146 | SH_308340 p46 | 계약자 본인 내방 시 대리인 내방 시(배우자, 직계가족 포함)  ① 계약자 신분증 : 주민
   3. 0.5193 | GUIDE_SEOUL_HLPA_2020 p8 | 두 번째는 ‘전월세 상한제’야, 보통 집주인이 계약만기 때 보증금이나 월세를 올려 받는 경
   4. 0.5269 | GUIDE_SEOUL_HLPA_2020 p35 | Q14 집주인이 전세 계약을 월세 계약으로 바꾸자고 해요 요즘 뉴스에 보니까 집주인들이 전
   5. 0.5369 | SH_299109 p20 | 홈페이지에 별도 게시합니다.  8  계약일시·장소 및 구비서류  ▣ 계약장소 : 서울주택도
질문: 이사 나갈 때 원상복구는 어디까지 해야 하나요?
  [jh_500]


   1. 0.5871 | YOUTH_6612 p27 | 다.  - 입주자 퇴거 시 세대 주거전용 부분 시설이 임차인의 고의 또는 과실, 부주의로 
   2. 0.6003 | GUIDE_SEOUL_HLPA_2020 p129 | 128 부록  이사시기불일치대출 관련 자주하는 질문  1. 이사시기불일치대출이란?  세입자
   3. 0.6024 | YOUTH_6613 p9 | - 입주자 퇴거 시 세대 주거전용 부분 시설이 임차인의 고의 또는 과실, 부주의로 인해 파
   4. 0.6091 | YOUTH_6642 p9 | - 입주자 퇴거 시 세대 주거 전용 부분 시설이 임차인의 고의 또는 과실, 부주의로 인해 
   5. 0.6153 | SH_304925 p28 | 보호대상 ◯ 보호대상  한부모가족  증명서(발급가능여부만  확인,  제출  불필요) 한부모
  [jh_1000]


   1. 0.5433 | YOUTH_6623 p1 | | 원상복구 계약 전 | 입주자 퇴거 시세대주거전용부분시설이임차인의고의또는과실, 부주의로인
   2. 0.5584 | YOUTH_6607 p1 | | 원상복구 계약 전 | 입주자 퇴거 시 세대 주거전용 부분 시설이임차인의 고의 또는 과실
   3. 0.5584 | YOUTH_6574 p1 | | 원상복구 계약 전 | 입주자 퇴거 시 세대 주거전용 부분 시설이임차인의 고의 또는 과실
   4. 0.5584 | YOUTH_6614 p1 | | 원상복구 계약 전 | 입주자 퇴거 시 세대 주거전용 부분 시설이임차인의 고의 또는 과실
   5. 0.5584 | YOUTH_6566 p1 | | 원상복구 계약 전 | 입주자 퇴거 시 세대 주거전용 부분 시설이임차인의 고의 또는 과실
질문: 행복주택 청년 자격이 어떻게 되나요?
  [jh_500]


   1. 0.3396 | SH_309337 p40 | 40 2026년 2차 행복주택 입주자모집  ▶ 청년
   2. 0.3552 | SH_304864 p41 | 2026년 1차 행복주택 입주자모집 41  ▶ 청년
   3. 0.4515 | SH_304864 p56 | 56 2026년 1차 행복주택 입주자모집
   4. 0.4520 | SH_304864 p66 | 66 2026년 1차 행복주택 입주자모집
   5. 0.4530 | SH_304864 p22 | 22 2026년 1차 행복주택 입주자모집
  [jh_1000]


   1. 0.4452 | SH_309337 p5 | □  2026년 2차 행복주택은 입주자모집공고일(2026. 08. 28.) 현재 대학생 ·
   2. 0.4515 | SH_304864 p56 | 56 2026년 1차 행복주택 입주자모집
   3. 0.4520 | SH_304864 p66 | 66 2026년 1차 행복주택 입주자모집
   4. 0.4582 | SH_309337 p56 | 56 2026년 2차 행복주택 입주자모집
   5. 0.4588 | SH_304864 p58 | 58 2026년 1차 행복주택 입주자모집
질문: 전세사기 안 당하려면 계약 전에 뭘 확인해야 하나요?
  [jh_500]


   1. 0.4766 | GUIDE_SEOUL_CASES p23 | 전세 계약하려는 집이 매매 진행 중입니다.  누구와 계약해야 할까요?  02  대리인 계약
   2. 0.4784 | SH_308340 p46 | ‘여권정보증명서’를 지참하여야 함  ‘여권정보증명서’를 지참하여야 함  ※  계약금을 납부
   3. 0.4848 | GUIDE_SEOUL_CASES p87 | Q  융자없는 신축 오피스텔이나 빌라 임차계약, 안전할까요?  A  최근 신축 다세대주택 
   4. 0.4903 | GUIDE_SEOUL_CASES p87 | ※ 참고 ‘전세보증보험은 무조건 가입이 되는 것’으로 생각하시는 분들이 있습니다. 그러나 
   5. 0.4946 | YOUTH_6558 p2 | ■ 입주자 세대 확인은 계약 체결 전 진행됩니다.
  [jh_1000]


   1. 0.4498 | GUIDE_SEOUL_CASES p87 | Q  융자없는 신축 오피스텔이나 빌라 임차계약, 안전할까요?  A  최근 신축 다세대주택 
   2. 0.4747 | GUIDE_SEOUL_CASES p23 | 전세 계약하려는 집이 매매 진행 중입니다.  누구와 계약해야 할까요?  02  대리인 계약
   3. 0.5067 | GUIDE_SEOUL_CASES p22 | CHAPTER  2-1  실제 상담사례로 살펴보는 전월세계약 필수 상식 계약하기: 계약서 
   4. 0.5234 | GUIDE_SEOUL_CASES p68 | CHAPTER  2-3  실제 상담사례로 살펴보는 전월세계약 필수 상식 계약 종료: 계약 
   5. 0.5261 | YOUTH_6613 p22 | 7 계약안내  ■ 계약 전 세대 확인 방문 및 전자계약 일시  | 구분 | 방문일자 | 방


## 6단계 기록

- 결과 해석과 표는 `실험기록.md` 7절에 적는다.